- setup an optimal control problem to reach a variety of target points
- check that you can stay within the allowed area after the target point was reached to show recursive feasibility

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

from dmpe.data_management import DataPaths
from dmpe.evaluation.plotting_utils import plot_sequence
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results
from dmpe.models.models import NeuralEulerODECartpole
from dmpe.models.model_utils import simulate_ahead_with_env

In [ ]:
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.utils.density_estimation import build_grid
from dmpe.models.model_utils import simulate_ahead_with_env

from dmpe.utils.reachable_set import approximate_reachable_set
from dmpe.utils.control_invariant_set import approximate_control_invariant_set

In [ ]:
env, penalty_function, featurize, _ = setup_pendulum_env()
key = jax.random.PRNGKey(0)
points_per_dim = 16
sequence_length = 50
n_starts = 100
n_opt_steps = 100

tolerance=1e-1


chosen_actions_rf, loss_map_rf, target_observations_rf, proposed_actions_rf = approximate_reachable_set(
    env,
    penalty_function,
    featurize,
    key=key,
    points_per_dim=points_per_dim,
    sequence_length=sequence_length,
    n_starts=n_starts,
    n_opt_steps=n_opt_steps,
)
plt.contourf(
    target_observations_rf.reshape((points_per_dim, points_per_dim, -1))[..., 0],
    target_observations_rf.reshape((points_per_dim, points_per_dim, -1))[..., 1],
    jnp.abs(loss_map_rf),
)
plt.show()

chosen_actions_ci, loss_map_ci, init_observations_ci, proposed_actions_ci = approximate_control_invariant_set(
    env,
    penalty_function,
    key,
    points_per_dim,
    sequence_length,
    n_starts,
    n_opt_steps=n_opt_steps
)
plt.contourf(
    init_observations_ci.reshape((points_per_dim, points_per_dim, -1))[..., 0],
    init_observations_ci.reshape((points_per_dim, points_per_dim, -1))[..., 1],
    (loss_map_ci == 0)
)
plt.show()

plt.contourf(
    init_observations_ci.reshape((points_per_dim, points_per_dim, -1))[..., 0],
    init_observations_ci.reshape((points_per_dim, points_per_dim, -1))[..., 1],
    jnp.logical_and((loss_map_ci == 0), (jnp.abs(loss_map_rf) < tolerance)),
)
plt.show()

## safe reachability - numerical approximation:

- create a single runnable function for safe reachability and control invariant set computations!

In [ ]:
from dmpe.utils.reachable_set import approximate_reachable_set

In [ ]:
env, penalty_function, featurize, _ = setup_pendulum_env()
key = jax.random.PRNGKey(0)
points_per_dim = 16
sequence_length = 300
n_starts = 100
n_opt_steps = 10_000

chosen_actions, loss_map, target_observations, proposed_actions = approximate_reachable_set(
    env,
    penalty_function,
    featurize,
    key=key,
    points_per_dim=points_per_dim,
    sequence_length=sequence_length,
    n_starts=n_starts,
    n_opt_steps=n_opt_steps,
)

In [ ]:
plt.contourf(
    target_observations.reshape((points_per_dim, points_per_dim, -1))[..., 0],
    target_observations.reshape((points_per_dim, points_per_dim, -1))[..., 1],
    jnp.abs(loss_map),
)
plt.show()

tolerance = 1e-1

plt.contourf(
    target_observations.reshape((points_per_dim, points_per_dim, -1))[..., 0],
    target_observations.reshape((points_per_dim, points_per_dim, -1))[..., 1],
    jnp.abs(loss_map) < tolerance,
)
plt.show()

fig, ax = plt.subplots(1,1, figsize=(3,10), sharey=True)
plt.imshow((jnp.abs(loss_map) < tolerance).T, cmap="plasma",  origin="lower", extent=[-np.pi, +np.pi, -10, 10], interpolation="nearest")
plt.show()

In [ ]:
n_targets = chosen_actions.shape[0]
for i in jnp.arange(0, n_targets, 12):
    observations, _ = simulate_ahead_with_env(
        env,
        init_obs,
        env.generate_state_from_observation(init_obs, env.env_properties),
        chosen_actions[i]
    )

    fig, axs = plt.subplots(nrows=1, ncols=4, figsize=(18, 6))
    
    axs[0].plot(observations[:, 0], observations[:, 1], "b.")
    axs[0].plot(target_observations[i, 0], target_observations[i, 1], "r.")
    axs[0].grid(True)

    axs[1].plot(jnp.arange(0, observations.shape[0]), jnp.linalg.norm(observations - target_observations[i][None], axis=-1))
    axs[1].grid(True)

    axs[2].plot(observations[:, 0])
    axs[2].plot(jnp.arange(0, observations.shape[0]), jnp.ones(observations.shape[0]) * target_observations[i, 0], "r")
    axs[2].grid(True)

    axs[3].plot(observations[:, 1])
    axs[3].plot(jnp.arange(0, observations.shape[0]), jnp.ones(observations.shape[0]) * target_observations[i, 1], "r")
    axs[3].grid(True)
    
    fig.tight_layout()
    plt.show()

## recursive feasibility (control invariant set) - numerical approximation:

In [ ]:
from dmpe.utils.control_invariant_set import approximate_control_invariant_set

### Pendulum:

In [ ]:
chosen_actions_ci, loss_map_ci, init_observations_ci, proposed_actions_ci = approximate_control_invariant_set(
    env,
    penalty_function,
    key,
    points_per_dim,
    sequence_length,
    n_starts,
    n_opt_steps=
)

In [ ]:
# env, penalty_function, featurize, _ = setup_pendulum_env()


# points_per_dim = 100


# key = jax.random.PRNGKey(0)
# proposed_actions = jax.random.uniform(key=jax.random.PRNGKey(0), shape=(init_observations.shape[0], 100, 100, 1), minval=-1, maxval=1)

# lr = optax.schedules.exponential_decay(
#     init_value=1e-1,
#     transition_steps=1_000,
#     transition_begin=0,
#     decay_rate=0.1,
#     end_value=1e-3,
# )

# optimizer = optax.adam(lr)

# chosen_actions, loss = eqx.filter_vmap(
#     rf_opt_actions, in_axes=(0, 0, None, None, None, None)
# )(
#     proposed_actions, init_observations, penalty_function, env, optimizer, 1_000
# )

In [ ]:
plt.contourf(
    init_observations_ci.reshape((points_per_dim, points_per_dim, -1))[..., 0],
    init_observations_ci.reshape((points_per_dim, points_per_dim, -1))[..., 1],
    (loss_map_ci != 0)
)

In [ ]:
plt.contourf(
    init_observations_ci.reshape((points_per_dim, points_per_dim, -1))[..., 0],
    init_observations_ci.reshape((points_per_dim, points_per_dim, -1))[..., 1],
    jnp.logical_and((loss_map_ci == 0), (jnp.abs(loss_map) < tolerance)),
)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(3,10), sharey=True)
plt.imshow((loss == 0).reshape((points_per_dim,points_per_dim)).T, cmap="plasma",  origin="lower", extent=[-np.pi, +np.pi, -10, 10], interpolation="nearest")

### Cartpole:

In [ ]:
env, penalty_function, featurize, _ = setup_cart_pole_env()

In [ ]:
points_per_dim = 20
dim = 4
init_observations = build_grid(dim, -1, 1, points_per_dim)

key = jax.random.PRNGKey(0)
proposed_actions = jax.random.uniform(key=jax.random.PRNGKey(0), shape=(init_observations.shape[0], 3, 100, 1), minval=-1, maxval=1)

In [ ]:
init_observations.shape

In [ ]:
lr = optax.schedules.exponential_decay(
    init_value=1e-1,
    transition_steps=1_000,
    transition_begin=0,
    decay_rate=0.1,
    end_value=1e-3,
)

optimizer = optax.adam(lr)

In [ ]:
chosen_actions, loss = eqx.filter_vmap(
    rf_opt_actions, in_axes=(0, 0, None, None, None, None)
)(
    proposed_actions, init_observations, penalty_function, env, optimizer, 1_000
)

In [ ]:
safe = (loss == 0).reshape([points_per_dim] * dim)

In [ ]:
labels = env.obs_description
n_features = 4
fig, axs = plt.subplots(nrows=n_features, ncols=n_features, figsize=(9, 9), sharex=True, sharey=True)

feature_indices = jnp.arange(0, n_features, 1).tolist()

for i in range(n_features):
    for j in range(n_features):

        axs[j, i].grid(True)
        axs[j, i].set_xlim(-1.1, 1.1)
        axs[j, i].set_ylim(-1.1, 1.1)

        reduction_indices = [f_idx for f_idx in feature_indices if not (f_idx == i or f_idx == j)]
        if len(reduction_indices) == n_features - 1:
            continue

        any_safe = jnp.any(safe, axis=tuple(reduction_indices))

        if i < j:
            any_safe = jnp.transpose(any_safe)

        axs[j, i].imshow(any_safe, origin="lower", extent=[-1, 1, -1, 1])
        axs[j, 0].set_ylabel(labels[j])

    axs[-1, i].set_xlabel(labels[i])
fig.tight_layout()

In [ ]:
any_safe = jnp.any(safe, axis=(0,1,2))
any_safe

## Stuff

In [ ]:
for i in range(init_observations.shape[0]):
    
    observations, _ = simulate_ahead_with_env(
        env,
        init_observations[i],
        env.generate_state_from_observation(init_observations[i], env.env_properties),
        proposed_actions[i, 0]
    )
    print(penalty_function(observations, proposed_actions[i, 0]))
    plot_sequence(observations, proposed_actions[i, 0], env.tau, env.obs_description, env.action_description)
    plt.show()

    print(loss[i])

    observations, _ = simulate_ahead_with_env(
        env,
        init_observations[i],
        env.generate_state_from_observation(init_observations[i], env.env_properties),
        chosen_actions[i]    
    )
    plot_sequence(observations, chosen_actions[i], env.tau, env.obs_description, env.action_description)
    plt.show()
    print("####")

In [ ]:
observations, _ = simulate_ahead_with_env(
    env,
    init_obs,
    env.generate_state_from_observation(init_obs, env.env_properties),
    proposed_actions[0]
)
print(penalty_function(observations, proposed_actions[0]))

plot_sequence(observations, proposed_actions[0], env.tau, env.obs_description, env.action_description)
plt.show()

###

chosen_actions, loss = optimize_actions_multistart(proposed_actions, init_obs, penalty_function, env, optimizer, 200)


###

print(loss)
observations, _ = simulate_ahead_with_env(
    env,
    init_obs,
    env.generate_state_from_observation(init_obs, env.env_properties),
    chosen_actions    
)
plot_sequence(observations, chosen_actions, env.tau, env.obs_description, env.action_description)
plt.show()

In [ ]:
from dmpe.utils.density_estimation import build_grid
from dmpe.evaluation.model_evaluation import EnvWrapper

In [ ]:
points_per_dim = 300
dim = 3
grid = build_grid(dim, -1, 1, points_per_dim)

In [ ]:
grid = grid.reshape(([points_per_dim] * dim + [-1]))

In [ ]:
safe_set = jnp.ones((points_per_dim, points_per_dim), dtype=bool)
initial_mask = jnp.zeros((points_per_dim, points_per_dim), dtype=bool)
initial_mask = initial_mask.at[points_per_dim//2, points_per_dim//2].set(True)
initial_mask

In [ ]:
wrapped_env = EnvWrapper(env, featurize=lambda x: x)

In [ ]:
def simulate_step(wrapped_env, grid, env):
    obs = grid[..., :-env.action_dim]
    act = grid[..., -env.action_dim:]
    
    out = eqx.filter_vmap(
        eqx.filter_vmap(
            eqx.filter_vmap(
                wrapped_env.step, in_axes=(0, 0, None)
            ), in_axes=(0, 0, None)
        ), in_axes=(0, 0, None)
    )(
        obs, act, env.tau
    )
    return out

def quantize_states_to_grid_batch(x, grid):

    theta_grid = grid[:, 0, 0]
    omega_grid = grid[0, :, 1]
    step_theta = theta_grid[1] - theta_grid[0]
    step_omega = omega_grid[1] - omega_grid[0]

    idx_theta = jnp.round((x[..., 0] - theta_grid[0]) / step_theta)
    idx_omega = jnp.round((x[..., 1] - omega_grid[0]) / step_omega)

    idx_theta = jnp.clip(idx_theta, 0, len(theta_grid) - 1).astype(jnp.int32)
    idx_omega = jnp.clip(idx_omega, 0, len(omega_grid) - 1).astype(jnp.int32)

    return jnp.stack([idx_theta, idx_omega], axis=-1)  # shape (5,5,5,2)

In [ ]:
mask = deepcopy(initial_mask)
plt.imshow(mask)
plt.show()

for i in tqdm(range(100)):

    out = simulate_step(
        wrapped_env, grid, env
    )    
    new_reachable = quantize_states_to_grid_batch(out, grid[:, :, 0, :2])[mask]
    new_reachable_mask = jnp.zeros(mask.shape)
    
    for r in jnp.squeeze(new_reachable):
        new_reachable_mask = new_reachable_mask.at[r[0], r[1]].set(True)
    
    mask = jnp.logical_or(mask, new_reachable_mask)
plt.imshow(mask)
plt.show()

In [ ]:
new_reachable

In [ ]:
import optimistix

In [ ]:
def loss_function(actions, state_target):
    state, target = state_target
    
    init_obs = env.generate_observation(state, env.env_properties)
    observations, _ = simulate_ahead_with_env(env, init_obs, state, actions)
    #observations, _, _ = env.sim_ahead(state, actions, env.env_properties, env.tau, env.tau)
    loss = jnp.mean(observations - target[None])**2
    penalties = penalty_function(observations, actions)
    return loss + penalties

# gradient_function = eqx.filter_grad(loss_function)

target=jnp.array([0.0, 0.0])
#proposed_actions = jnp.ones((2000, 1)) #jnp.concatenate([jnp.ones((50, 1)) * (-1)**i for i in range(20)])


proposed_actions = jax.random.uniform(key=jax.random.PRNGKey(0), shape=(2000, 1), minval=-1, maxval=1)

obs, state = env.reset(env.env_properties)

observations, _ = simulate_ahead_with_env(env, obs, state, proposed_actions)
plt.plot(observations[:, 0], label="theta")
plt.plot(observations[:, 1], label="omega")
plt.grid(True)
plt.show()

plt.plot(proposed_actions)
plt.grid(True)
plt.show()

print("initial_loss: ", loss_function(proposed_actions, (state, target)))

solver = optimistix.BFGS(rtol=1e-10, atol=1e-10)
sol = optimistix.minimise(
    loss_function, solver, proposed_actions, args=(state, target)
)

observations, _ = simulate_ahead_with_env(env, obs, state, sol.value)
plt.plot(observations[:, 0], label="theta")
plt.plot(observations[:, 1], label="omega")
plt.grid(True)
plt.show()

plt.plot(sol.value)
plt.grid(True)
plt.show()

print("final_loss: ", loss_function(sol.value, (state, target)))

In [ ]:
sol.stats

In [ ]:
sol.state.y_eval

In [ ]:
@eqx.filter_jit
def optimize_actions(proposed_actions, state, target, optimizer, n_opt_steps):
    opt_state = optimizer.init(proposed_actions)

    def body_fun(i, carry):
        proposed_actions, opt_state = carry
        grad = gradient_function(
            proposed_actions, state, target,
        )
        updates, opt_state = optimizer.update(grad, opt_state, proposed_actions)
        proposed_actions = optax.apply_updates(proposed_actions, updates)

        return (proposed_actions, opt_state)

    proposed_actions, _ = jax.lax.fori_loop(0, n_opt_steps, body_fun, (proposed_actions, opt_state))
    return proposed_actions

In [ ]:
def loss_function(actions, state, target):
    init_obs = env.generate_observation(state, env.env_properties)
    observations, _ = simulate_ahead_with_env(env, init_obs, state, actions)
    #observations, _, _ = env.sim_ahead(state, actions, env.env_properties, env.tau, env.tau)
    loss = jnp.mean(observations[-100:] - target[None])**2
    penalties = penalty_function(observations, actions) * 1e3
    return loss + penalties

gradient_function = eqx.filter_grad(loss_function)

lr = optax.schedules.exponential_decay(
    init_value=1e-1,
    transition_steps=1000,
    transition_begin=0,
    decay_rate=0.1,
    end_value=1e-3,
)

optimizer = optax.adam(lr)
# proposed_actions = jax.random.uniform(
#     key=jax.random.PRNGKey(0), 
#     shape=(100, 500, 1),
#     minval=-1,
#     maxval=1
# )
proposed_actions = jnp.concatenate([jnp.ones((100, 1)) * (-1)**i for i in range(5)])[None]

target=jnp.array([0.0, 0.0])
obs, state = env.reset(env.env_properties)

#####

observations, _ = simulate_ahead_with_env(env, obs, state, proposed_actions[0])

plt.plot(observations[:, 0], label="theta")
plt.plot(observations[:, 1], label="omega")
plt.grid(True)
plt.show()

plt.plot(proposed_actions[0])
plt.grid(True)
plt.show()

#########

actions = eqx.filter_vmap(optimize_actions, in_axes=(0, None, None, None, None))(
    proposed_actions, state, target, optimizer, 2000)

losses = eqx.filter_vmap(loss_function, in_axes=(0, None, None))(actions, state, target)
best_idx = jnp.argmin(losses)

########

observations, _ = simulate_ahead_with_env(env, obs, state, actions[best_idx])

plt.plot(observations[:, 0], label="theta")
plt.plot(observations[:, 1], label="omega")
plt.grid(True)
plt.show()

plt.plot(actions[best_idx])
plt.grid(True)
plt.show()

In [ ]:
def loss_function(actions, state, target):
    init_obs = env.generate_observation(state, env.env_properties)
    observations, _ = simulate_ahead_with_env(env, init_obs, state, actions)
    #observations, _, _ = env.sim_ahead(state, actions, env.env_properties, env.tau, env.tau)
    loss = jnp.mean(observations[-100:] - target[None])**2
    penalties = penalty_function(observations, actions) * 1e3
    return loss + penalties

gradient_function = eqx.filter_grad(loss_function)
optimizer = optax.adam(1e-2)

In [ ]:
obs, state = env.reset(env.env_properties)

observations = [obs]
states = [state]
actions = []

proposed_actions = jnp.concatenate([jnp.ones((25, 1)) * (-1)**i for i in range(4)])

for i in tqdm(range(200)):
    opt_state = optimizer.init(proposed_actions)
    for j in range(100):
        grad = gradient_function(proposed_actions, state, target)
        updates, opt_state = optimizer.update(grad, opt_state, proposed_actions)
        proposed_actions = optax.apply_updates(proposed_actions, updates)

    action = proposed_actions[0, :]
    obs, state = env.step(state, action, env.env_properties)
    
    observations.append(obs)
    states.append(state)
    actions.append(action)

In [ ]:
plt.plot(observations)
plt.grid(True)
plt.show()

plt.plot(actions)
plt.grid(True)
plt.show()

plt.scatter(jnp.array(observations)[:, 0], jnp.array(observations)[:, 1], s=2)
plt.grid(True)
plt.show()